# Module 33 — Stretch: RLHF/DPO Concepts

Module 32's instruction fine-tuning teaches a model *a* correct behavior
per example. Production conversational models add one more step:
**preference alignment** — given two candidate responses to the same
prompt, one preferred over the other, nudge the model toward the
preferred kind of response in general, not just toward memorizing single
correct answers.

**RLHF** (Reinforcement Learning from Human Feedback) does this by
training a separate *reward model* on human preference judgments, then
running a reinforcement learning loop (PPO) that optimizes the policy
against that learned reward — real, but genuinely heavy machinery: two
models, an RL training loop, and all of RL's usual stability challenges.

**DPO** (Direct Preference Optimization, Rafailov et al. 2023) is this
project's stretch target instead: it proves the *same* alignment objective
reduces to a single, simple, closed-form loss computed directly on
preference pairs — no separate reward model, no RL loop at all. That
tractability is exactly why it's implementable here.

## 1. The DPO loss

For a prompt with a preferred response `y_w` ("winning") and a dispreferred
response `y_l` ("losing"), DPO compares each response's log-probability
under the **policy** being trained against the same response's
log-probability under a **frozen reference model** (a copy of the model
*before* this stage — Module 32's SFT checkpoint, in a real pipeline):

```
margin = (log pi(y_w|x) - log pi_ref(y_w|x)) - (log pi(y_l|x) - log pi_ref(y_l|x))
loss   = -log(sigmoid(beta * margin))
```

Maximizing `margin` means: the policy increases its relative preference
(vs. the frozen reference) for the winning response more than for the
losing one. `beta` controls how aggressively to enforce this.

## 2. Setup: policy + frozen reference (same architecture as Modules 18/32, not re-explained)

In [ ]:
import copy
import math

import torch
import torch.nn as nn
import torch.nn.functional as F

CORPUS = """Aether and Lumine are twins known as the Traveler. They came from another world and lost each other upon arrival in Teyvat. Paimon found Aether floating near Mondstadt and decided to travel together. Mondstadt is called the City of Freedom, and the wind blows gently across its hills. Klee loves to explore the city and often causes small explosions with her bombs. Diluc runs the Dawn Winery outside the city walls. Kaeya works at the Knights of Favonius and enjoys teasing Diluc. Jean leads the Knights of Favonius with great responsibility. Barbara sings songs at the church and heals the sick. Venti wanders the city playing his lyre and humming old songs. Amber flies her glider over the fields, scouting for trouble."""

PROMPT_MARKER = "\n### Answer:"
questions_and_good_answers = [
    ("### Question: Who is Klee?", " Klee loves to explore the city and often causes small explosions with her bombs."),
    ("### Question: Who is Zhongli?", " Zhongli walks slowly through the streets, remembering old stories."),
    ("### Question: Who is Venti?", " Venti wanders the city playing his lyre and humming old songs."),
]
# a dispreferred answer per question: a real, grammatical sentence - just the wrong one
bad_answers = [
    " Diluc runs the Dawn Winery outside the city walls.",
    " Klee loves to explore the city and often causes small explosions with her bombs.",
    " Barbara sings songs at the church and heals the sick.",
]

extra_chars = "".join(q + PROMPT_MARKER + a for q, a in questions_and_good_answers) + "".join(bad_answers)
torch.manual_seed(42)
chars = sorted(set(CORPUS) | set(extra_chars))
stoi = {ch: i for i, ch in enumerate(chars)}
vocab_size = len(chars)


def scaled_dot_product_attention(Q, K, V, causal=True):
    d_k = Q.shape[-1]
    scores = (Q @ K.transpose(-2, -1)) / math.sqrt(d_k)
    if causal:
        seq_len_q, seq_len_k = scores.shape[-2], scores.shape[-1]
        mask = torch.triu(torch.ones(seq_len_q, seq_len_k, device=scores.device), diagonal=1).bool()
        scores = scores.masked_fill(mask, float("-inf"))
    weights = F.softmax(scores, dim=-1)
    return weights @ V, weights

def split_heads(t, num_heads):
    *batch_dims, seq_len, d_model = t.shape
    d_k = d_model // num_heads
    return t.view(*batch_dims, seq_len, num_heads, d_k).transpose(-3, -2)

def merge_heads(t):
    *batch_dims, num_heads, seq_len, d_k = t.shape
    return t.transpose(-3, -2).contiguous().view(*batch_dims, seq_len, num_heads * d_k)

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.num_heads = num_heads
        self.Wq = nn.Linear(d_model, d_model, bias=False)
        self.Wk = nn.Linear(d_model, d_model, bias=False)
        self.Wv = nn.Linear(d_model, d_model, bias=False)
        self.Wo = nn.Linear(d_model, d_model, bias=False)
    def forward(self, x, causal=True):
        Q, K, V = self.Wq(x), self.Wk(x), self.Wv(x)
        Qh, Kh, Vh = split_heads(Q, self.num_heads), split_heads(K, self.num_heads), split_heads(V, self.num_heads)
        out, _ = scaled_dot_product_attention(Qh, Kh, Vh, causal=causal)
        return self.Wo(merge_heads(out))

class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff=None):
        super().__init__()
        d_ff = d_ff or 4 * d_model
        self.net = nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model))
    def forward(self, x):
        return self.net(x)

class TransformerBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff=None):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = MultiHeadAttention(d_model, num_heads)
        self.ln2 = nn.LayerNorm(d_model)
        self.ffn = FeedForward(d_model, d_ff)
    def forward(self, x):
        x = x + self.attn(self.ln1(x), causal=True)
        x = x + self.ffn(self.ln2(x))
        return x

class NanoGPT(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, num_layers, max_seq_len, d_ff=None):
        super().__init__()
        self.max_seq_len = max_seq_len
        self.token_embed = nn.Embedding(vocab_size, d_model)
        self.pos_embed = nn.Embedding(max_seq_len, d_model)
        self.blocks = nn.ModuleList([TransformerBlock(d_model, num_heads, d_ff) for _ in range(num_layers)])
        self.ln_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)
        self.head.weight = self.token_embed.weight
    def forward(self, idx):
        seq_len = idx.shape[-1]
        positions = torch.arange(seq_len, device=idx.device)
        x = self.token_embed(idx) + self.pos_embed(positions)
        for block in self.blocks:
            x = block(x)
        x = self.ln_f(x)
        return self.head(x)


MAX_SEQ_LEN = 150
device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(42)
# In a real pipeline, policy_model starts as a COPY of Module 32\'s SFT checkpoint.
# This stretch module verifies the DPO mechanism itself, so it starts from a fresh model.
policy_model = NanoGPT(vocab_size, d_model=64, num_heads=4, num_layers=4, max_seq_len=MAX_SEQ_LEN).to(device)

ref_model = copy.deepcopy(policy_model)
for p in ref_model.parameters():
    p.requires_grad = False
ref_model.eval()
print("Policy and frozen reference model created (identical weights at the start).")

## 3. Preference pairs, and computing response log-probability

Same loss-masking idea as Module 32 (only response tokens count), but here
used to compute a *log-probability*, not a loss directly.

In [ ]:
def encode(s):
    return [stoi[c] for c in s]

def build_example(question, answer):
    prompt_text = question + PROMPT_MARKER
    full_ids = encode(prompt_text + answer)
    prompt_len = len(encode(prompt_text))
    return full_ids, prompt_len

def response_logprob(model, full_ids, prompt_len):
    ids = torch.tensor([full_ids], device=device)
    input_ids, target_ids = ids[:, :-1], ids[:, 1:]
    log_probs = F.log_softmax(model(input_ids), dim=-1)
    token_logprobs = log_probs.gather(-1, target_ids.unsqueeze(-1)).squeeze(-1)
    L = len(full_ids) - 1
    resp_mask = torch.tensor([[1.0 if (t + 1) >= prompt_len else 0.0 for t in range(L)]], device=device)
    return (token_logprobs * resp_mask).sum(dim=-1)


triples = [
    (*build_example(q, good), *build_example(q, bad))
    for (q, good), bad in zip(questions_and_good_answers, bad_answers)
]
print(f"{len(triples)} preference triples (prompt, preferred response, dispreferred response) built.")

## 4. The DPO training loop, with a theoretical sanity check

At step 0, the policy and reference are identical, so `margin = 0` for
every pair, and the loss should be exactly `-log(sigmoid(0)) = log(2) ≈
0.693` — a concrete, checkable prediction, not just "some number."

In [ ]:
BETA = 0.1

def dpo_loss_and_margin():
    total_loss, total_margin = 0.0, 0.0
    for good_ids, good_plen, bad_ids, bad_plen in triples:
        policy_good = response_logprob(policy_model, good_ids, good_plen)
        policy_bad = response_logprob(policy_model, bad_ids, bad_plen)
        with torch.no_grad():
            ref_good = response_logprob(ref_model, good_ids, good_plen)
            ref_bad = response_logprob(ref_model, bad_ids, bad_plen)

        margin = (policy_good - ref_good) - (policy_bad - ref_bad)
        loss = -F.logsigmoid(BETA * margin).mean()
        total_loss = total_loss + loss
        total_margin += margin.item()
    return total_loss / len(triples), total_margin / len(triples)


initial_loss, initial_margin = dpo_loss_and_margin()
print(f"initial loss: {initial_loss.item():.4f}  (expected: ln(2) = {math.log(2):.4f})")
print(f"initial margin: {initial_margin:.4f}  (expected: 0, policy == reference)")
assert abs(initial_loss.item() - math.log(2)) < 1e-4
assert abs(initial_margin) < 1e-4
print("Confirmed: matches the theoretical prediction exactly before any training.")

ref_params_before = [p.clone() for p in ref_model.parameters()]
optimizer = torch.optim.AdamW(policy_model.parameters(), lr=1e-3)

for step in range(200):
    loss, margin = dpo_loss_and_margin()
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if step % 40 == 0:
        print(f"step {step:3d}   dpo loss {loss.item():.4f}   avg preference margin {margin:.2f}")

## 5. The reference model must never change

In [ ]:
for p_before, p_after in zip(ref_params_before, ref_model.parameters()):
    assert torch.equal(p_before, p_after)
print("Confirmed: the frozen reference model\'s weights are identical before and after DPO training.")
print("Only the policy model was updated - exactly what \'frozen reference\' is supposed to mean.")

## Recap

- DPO reduces preference alignment to one closed-form loss computed
  directly on (preferred, dispreferred) response pairs — no reward model,
  no RL loop.
- Verified the loss matches its exact theoretical value (`ln(2)`) when
  policy and reference are identical, then grew a large positive
  preference margin over 200 steps as the policy learned to favor the
  preferred responses.
- Confirmed the reference model's weights never changed — it exists purely
  as a fixed comparison point, per the algorithm's design.
- RLHF's reward-model-plus-PPO approach achieves a similar goal but adds
  substantial complexity (a second model, an RL training loop, RL's usual
  instability); DPO's appeal is doing without any of that.

This closes out the roadmap. Modules 01-18 built a transformer from
individual scalars up to a trained (if tiny) GPT; 19-29 added everything
needed to run that at production scale; 30-31 built and ran a real
~125M-parameter pretrain; 32-33 covered turning a raw pretrained model into
something aligned to instructions and preferences. Real-scale execution of
Module 31's pretraining run (on Colab Pro, at the full 2.5B-token budget)
is the next actual step outside this notebook series.